In [ ]:
import geopandas as gpd
land_gdf = gpd.read_file("../data/processed/land_data.gpkg")


In [ ]:
dataset_for_merging = gpd.read_file("../data/processed/corridor_centerlines_ridge.gpkg") # or "corridor_centerlines_skeletionization.gpkg"
gpkg_file = "ridge_output.gpkg"  # or "corridor_output.gpkg"


In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString
from shapely.ops import nearest_points

MAX_PROBE_DISTANCE = 200    # archipelago passages can be wide
CHANNEL_THRESHOLD = 120     # allows detection up to ~240m total width
SYMMETRY_RATIO_MIN = 0.2    # archipelago geometry is irregular
NARROW_STRAIT_THRESHOLD = 0.3  # routes alternate open/confined

def _perpendicular_direction(route_geometry, distance: float):
    """
    Return a unit vector perpendicular to the route at a given distance
    along it.
    """
    delta = min(1.0, route_geometry.length * 0.01)
    p1 = route_geometry.interpolate(max(0, distance - delta))
    p2 = route_geometry.interpolate(min(route_geometry.length, distance + delta))

    dx = p2.x - p1.x
    dy = p2.y - p1.y
    norm = np.hypot(dx, dy)
    if norm == 0:
        return (1.0, 0.0)  # fallback for degenerate segments

    #  rotate tangent 90 degrees
    return (-dy / norm, dx / norm)


def _land_distance_along_ray(origin, direction, max_dist, land_sindex, land_geom):
    """
    Cast a ray from $origin in $direction and return the distance to the
    nearest land polygon, or $max_dist if nothing is hit.
    """
    tip_x = origin.x + direction[0] * max_dist
    tip_y = origin.y + direction[1] * max_dist
    ray = LineString([(origin.x, origin.y), (tip_x, tip_y)])

    candidate_indices = list(land_sindex.intersection(ray.bounds))
    if not candidate_indices:
        return max_dist

    nearby = land_geom.iloc[candidate_indices]
    hits = nearby[nearby.intersects(ray)]
    if hits.empty:
        return max_dist

    # Find the nearest intersection point along the ray
    min_dist = max_dist
    for geom in hits:
        nearest_pt = nearest_points(origin, geom)[1]
        d = origin.distance(nearest_pt)
        if d < min_dist:
            min_dist = d

    return min_dist


def calculate_channel_confinement(
    route_geometry,
    land: gpd.GeoDataFrame,
    intervals_meters: int = 20,
    max_probe_distance: float = MAX_PROBE_DISTANCE,
    channel_threshold: float = CHANNEL_THRESHOLD,
    symmetry_ratio_min: float = SYMMETRY_RATIO_MIN,
):
    """
    Evaluate whether a route passes through a constrained channel.
    """
    if route_geometry is None or route_geometry.is_empty:
        return {"confinement_score": 0.0, "mean_left_dist": np.nan,
                "mean_right_dist": np.nan, "mean_width": np.nan}

    length = route_geometry.length
    distances = np.linspace(0, length, max(2, int(length / intervals_meters) + 1))

    land_geom = land.geometry
    land_sindex = land.sindex

    left_dists, right_dists, confined_widths = [], [], []
    confined_count = 0

    for d in distances:
        pt = route_geometry.interpolate(d)
        perp = _perpendicular_direction(route_geometry, d)
        left_dir = (perp[0], perp[1])
        right_dir = (-perp[0], -perp[1])

        dL = _land_distance_along_ray(pt, left_dir, max_probe_distance, land_sindex, land_geom)
        dR = _land_distance_along_ray(pt, right_dir, max_probe_distance, land_sindex, land_geom)

        left_dists.append(dL)
        right_dists.append(dR)

        # A point can be confinded only if: land is present on two sides within the threshold &&  the two distances are balanced
        both_close = dL < channel_threshold and dR < channel_threshold
        symmetric = min(dL, dR) / max(dL, dR) >= symmetry_ratio_min if max(dL, dR) > 0 else False

        if both_close and symmetric:
            confined_count += 1
            confined_widths.append(dL + dR)

    n = len(distances)
    return {
        "confinement_score": confined_count / n,
        "mean_left_dist": float(np.mean(left_dists)),
        "mean_right_dist": float(np.mean(right_dists)),
        "mean_width": float(np.mean(confined_widths)) if confined_widths else np.nan,
    }

def _classify_corridor(route_type: str, confinement_score: float):
    is_narrow = confinement_score >= NARROW_STRAIT_THRESHOLD

    if route_type == "Two-Way":
        return "Narrow Two-Way Corridor" if is_narrow else "Open Two-Way Passage"
    elif "One-Way" in route_type:
        return "Narrow One-Way Corridor" if is_narrow else "Open One-Way Passage"
    else:
        return "Unclassified"

def evaluate_channel_confinement(
    candidates_input,
    land_input,
    output_gpkg_path: str,
    intervals_meters: int = 20,
    max_probe_distance: float = MAX_PROBE_DISTANCE,
    channel_threshold: float = CHANNEL_THRESHOLD,
    symmetry_ratio_min: float = SYMMETRY_RATIO_MIN,
):
    """
    Score each candidate route by whether it has land on both sides
    """
    candidates = (
        gpd.read_file(candidates_input)
        if isinstance(candidates_input, str)
        else candidates_input.copy()
    )
    land = (
        gpd.read_file(land_input)
        if isinstance(land_input, str)
        else land_input.copy()
    )

    required_columns = {"geometry", "type", "path_reliability_score"}
    missing = required_columns - set(candidates.columns)
    if missing:
        raise ValueError(f"candidates is missing required columns: {missing}")

    if candidates.crs is None:
        raise ValueError("candidates gdf has no CRS set.")
    if land.crs is None:
        raise ValueError("land gdf has no CRS set.")

    if candidates.crs != land.crs:
        land = land.to_crs(candidates.crs)

    results = candidates["geometry"].apply(
        lambda geom: calculate_channel_confinement(
            geom,
            land,
            intervals_meters=intervals_meters,
            max_probe_distance=max_probe_distance,
            channel_threshold=channel_threshold,
            symmetry_ratio_min=symmetry_ratio_min,
        )
    )

    candidates["confinement_score"] = results.apply(lambda r: r["confinement_score"])
    candidates["mean_left_dist"]    = results.apply(lambda r: r["mean_left_dist"])
    candidates["mean_right_dist"]   = results.apply(lambda r: r["mean_right_dist"])
    candidates["mean_width"]        = results.apply(lambda r: r["mean_width"])

    candidates["route_safety_index"] = (
        candidates["path_reliability_score"] * 0.5
        + candidates["confinement_score"] * 0.5
    )

    candidates["corridor_classification"] = candidates.apply(
        lambda row: _classify_corridor(row["type"], row["confinement_score"]),
        axis=1,
    )

    candidates = candidates.sort_values(by="route_safety_index", ascending=False)

    candidates.to_file(output_gpkg_path, layer="channel_confinement", driver="GPKG")
    print("Done.")

    return candidates

In [ ]:
final_df = evaluate_channel_confinement(
    candidates_input=skeleton,
    land_input=land_gdf,
    output_gpkg_path="../../data/processed/centerlines_cosntrained_skeletonization.gpkg"
)

In [ ]:
final_df_ridge = evaluate_channel_confinement(
    candidates_input=ridge,
    land_input=land_gdf,
    output_gpkg_path="../../data/processed/centerlines_cosntrained_ridge.gpkg"
)